# 01 · Clean and merge: Low Water Crossing Risk

Builds **one row per low water crossing** (156 crossings) with features from the other city and state datasets, then a first risk score and clusters.

**Output:** `data/processed/crossings.csv` (hand this to the briefs notebook) and `data/processed/risk_map.html`.

**Inputs:** put the downloaded CSVs in `data/raw/` with their original names; the loader matches them by the start of the name.

| File | Rows | How it joins to crossings |
| --- | --- | --- |
| LowWaterCrossings | 156 | the base table |
| StormInlets | 30,212 | by distance (x, y) |
| IMP_Drainage | 628 | by distance (INSIDE_X, INSIDE_Y) |
| Sanitary Sewer Overflows (TCEQ) | 56,545 statewide | no coordinates: Bexar County only, matched by creek name |
| StormChannels | 12,381 | **not used**: the CSV has no coordinates (see note at the end) |

In [ ]:
%pip install -q pyproj folium scipy scikit-learn

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from pyproj import Transformer

RAW = Path("data/raw")
OUT = Path("data/processed"); OUT.mkdir(parents=True, exist_ok=True)

def load(prefix):
    files = sorted(RAW.glob(prefix + "*.csv"))
    assert files, f"No file starting with {prefix!r} in {RAW.resolve()}"
    df = pd.read_csv(files[0], low_memory=False)
    df.columns = df.columns.str.strip()          # one inlet column name ends in a newline
    print(f"{prefix:42s} {df.shape}")
    return df

lwc     = load("LowWaterCrossings")
inlets  = load("StormInlets")
imp     = load("IMP_Drainage")
sso     = load("Texas_Commission_on_Environmental_Quality_-_Sanitary")

## 1. Coordinates

`x` and `y` are in **Texas State Plane South Central (EPSG:2278), in US feet**, so distances can be computed directly in feet. We also add latitude and longitude for the map.

In [ ]:
to_latlon = Transformer.from_crs(2278, 4326, always_xy=True)
lwc["lon"], lwc["lat"] = to_latlon.transform(lwc["x"].values, lwc["y"].values)

# sanity check: every crossing should fall inside the San Antonio area
assert lwc["lat"].between(29.1, 29.9).all() and lwc["lon"].between(-98.9, -98.1).all()
assert lwc["CartID"].is_unique
lwc[["Name", "CartID", "lat", "lon"]].head()

## 2. Clean the crossings

Pull out the creek name where the crossing's name includes one (for example *Rice Rd at Salado Creek*). Only some crossings name their creek; that limits the sewer overflow match in step 5.

In [ ]:
def creek_from_name(name):
    m = re.search(r"([A-Za-z]+\s+Creek)", name)
    return m.group(1).title() if m else None

lwc["Name"] = lwc["Name"].str.replace(r"\s+", " ", regex=True).str.strip()
lwc["creek"] = lwc["Name"].map(creek_from_name)
print(lwc["creek"].notna().sum(), "of", len(lwc), "crossings name a creek")
lwc["creek"].value_counts()

## 3. Storm inlets near each crossing

Inlets carry street runoff into the drainage system. For each crossing we count **active** inlets within a radius, and how many of them are in **Poor or Failed** condition. Condition `TBD` means not yet inspected.

In [ ]:
INLET_RADIUS_FT = 1500     # about 460 m; a judgment call, try 1000 and 2500

inl = inlets.dropna(subset=["x", "y"]).copy()
inl = inl[(inl["x"] > 0) & (inl["y"] > 0)]
inl["Condition"] = inl["Condition"].str.strip().str.title()
inl["Status"] = inl["Status"].str.strip().str.title()
print(inl["Status"].value_counts().to_dict())
print(inl["Condition"].value_counts().to_dict())

active = inl[inl["Status"] == "Active"]
tree = cKDTree(active[["x", "y"]].values)
near = tree.query_ball_point(lwc[["x", "y"]].values, r=INLET_RADIUS_FT)

cond = active["Condition"].values
lwc["inlets_active"]       = [len(ix) for ix in near]
lwc["inlets_poor_failed"]  = [int(np.isin(cond[ix], ["Poor", "Failed"]).sum()) for ix in near]
lwc["inlets_uninspected"]  = [int((cond[ix] == "Tbd").sum()) for ix in near]
lwc["dist_nearest_inlet_ft"] = tree.query(lwc[["x", "y"]].values)[0].round(0)
lwc[["Name", "inlets_active", "inlets_poor_failed", "inlets_uninspected", "dist_nearest_inlet_ft"]].describe()

## 4. Drainage maintenance projects (IMP) near each crossing

The IMP Drainage file is the city's **five-year plan** (fiscal years 2026 to 2030) for creek debris removal and channel restoration. `INSIDE_X/Y` is a point on each project's creek segment. Most date columns are 80%+ empty, so we use the fiscal year instead.

In [ ]:
IMP_RADIUS_FT = 2640       # half a mile; project segments are long

imp_c = imp.dropna(subset=["INSIDE_X", "INSIDE_Y"]).copy()
imp_c["Project Type"] = imp_c["Project Type"].str.strip()
imp_c["Name"] = imp_c["Name"].str.strip().str.title()
print(imp_c["Project Type"].value_counts().to_dict(), "| fiscal years:", sorted(imp_c["Fiscal Year"].unique()))

itree = cKDTree(imp_c[["INSIDE_X", "INSIDE_Y"]].values)
inear = itree.query_ball_point(lwc[["x", "y"]].values, r=IMP_RADIUS_FT)
fy, cost, ptype = imp_c["Fiscal Year"].values, imp_c["Estimated Total Cost"].fillna(0).values, imp_c["Project Type"].values

lwc["imp_projects_nearby"]  = [len(ix) for ix in inear]
lwc["imp_restoration_nearby"] = [int((ptype[ix] == "Restoration").sum()) for ix in inear]
lwc["imp_next_fiscal_year"] = [int(fy[ix].min()) if len(ix) else np.nan for ix in inear]
lwc["imp_planned_cost"]     = [round(float(cost[ix].sum())) for ix in inear]
lwc["dist_nearest_imp_ft"]  = itree.query(lwc[["x", "y"]].values)[0].round(0)
print((lwc["imp_projects_nearby"] > 0).sum(), "crossings have planned drainage work within half a mile")

## 5. Sewer overflows on the same creek

The TCEQ file covers all of Texas and has **no coordinates**. We keep Bexar County, standardize the receiving water body (it is typed many ways: *SALADO CREEK*, *salado creek*, *1906 LEON CREEK*) and count overflows into each crossing's creek since 2021. *Infiltration and inflow* is the cause most tied to heavy rain.

In [ ]:
b = sso[sso["County"].str.strip().str.upper() == "BEXAR"].copy()
b["start"] = pd.to_datetime(b["Start Date"], format="%b %d, %Y", errors="coerce")
b["gallons"] = pd.to_numeric(b["Amount"].str.replace(",", ""), errors="coerce")
b["cause"] = b["Cause"].str.strip().str.upper()
b["water"] = (b["Receiving Water Body"].fillna("").str.upper()
              .str.replace(r"^\d+\s+", "", regex=True).str.strip()
              .map(lambda s: creek_from_name(s.title()) if "CREEK" in s else None))
b = b[b["start"] >= "2021-01-01"]
print(len(b), "Bexar overflows since 2021 |", b["start"].isna().sum(), "unparsed dates")
print(b["water"].value_counts().head(12).to_dict())

by_creek = b.dropna(subset=["water"]).groupby("water").agg(
    sso_same_creek=("Incident Number", "count"),
    sso_rain_related=("cause", lambda c: int((c == "INFILTRATION AND INFLOW").sum())),
    sso_gallons=("gallons", "sum"))
lwc = lwc.merge(by_creek, left_on="creek", right_index=True, how="left", validate="many_to_one")
for c in ["sso_same_creek", "sso_rain_related", "sso_gallons"]:
    lwc[c] = lwc[c].fillna(0).astype(int)
lwc.loc[lwc["creek"].notna(), ["Name", "creek", "sso_same_creek", "sso_rain_related"]]

## 6. Check the merged table

One row per crossing, no rows lost, no missing values in the features.

In [ ]:
features = ["inlets_active", "inlets_poor_failed", "inlets_uninspected", "dist_nearest_inlet_ft",
            "imp_projects_nearby", "imp_restoration_nearby", "imp_planned_cost", "dist_nearest_imp_ft",
            "sso_same_creek", "sso_rain_related"]
assert len(lwc) == 156 and lwc["CartID"].is_unique
assert lwc[features].notna().all().all()
lwc[features].describe().round(1).T

## 7. First risk score and clusters

The score is deliberately simple so it can be explained on a slide. Each feature is converted to a z-score (how unusual it is compared with other crossings), then:

- **raises risk:** poor or failed inlets nearby, uninspected inlets nearby, distance to the nearest inlet, sewer overflows on the same creek
- **lowers risk:** more active inlets nearby, planned drainage work within half a mile

It ranks crossings by warning signs in the data. It is **not** a flood probability.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

z = pd.DataFrame(StandardScaler().fit_transform(lwc[features]), columns=features, index=lwc.index)
lwc["risk_score"] = (z["inlets_poor_failed"] + 0.5 * z["inlets_uninspected"] + z["dist_nearest_inlet_ft"]
                     + z["sso_same_creek"] - z["inlets_active"] - 0.5 * z["imp_projects_nearby"]).round(2)
lwc["risk_rank"] = lwc["risk_score"].rank(ascending=False, method="first").astype(int)

km = KMeans(n_clusters=4, n_init=10, random_state=0).fit(z)
lwc["cluster"] = km.labels_
print("Average features per cluster (name them from this):")
lwc.groupby("cluster")[features + ["risk_score"]].mean().round(1).T

In [ ]:
lwc.nsmallest(10, "risk_rank")[["risk_rank", "Name", "risk_score", "inlets_active",
                                  "inlets_poor_failed", "dist_nearest_inlet_ft", "imp_projects_nearby", "sso_same_creek"]]

## 8. Save and map

In [ ]:
cols = ["CartID", "Name", "creek", "lat", "lon", "x", "y"] + features + ["imp_next_fiscal_year", "risk_score", "risk_rank", "cluster"]
lwc[cols].sort_values("risk_rank").to_csv(OUT / "crossings.csv", index=False)
print("Saved", OUT / "crossings.csv", lwc[cols].shape)

In [ ]:
import folium
m = folium.Map(location=[lwc["lat"].mean(), lwc["lon"].mean()], zoom_start=10)
lo, hi = lwc["risk_score"].min(), lwc["risk_score"].max()
for r in lwc.itertuples():
    s = (r.risk_score - lo) / (hi - lo)
    folium.CircleMarker(
        [r.lat, r.lon], radius=4 + 9 * s, fill=True, fill_opacity=0.8, weight=1,
        color=f"#{int(220*s):02x}{int(140*(1-s)):02x}{int(200*(1-s)):02x}",
        tooltip=(f"#{r.risk_rank} {r.Name}<br>score {r.risk_score} · active inlets {r.inlets_active}"
                 f" · poor/failed {r.inlets_poor_failed} · drainage projects {r.imp_projects_nearby}")).add_to(m)
m.save(OUT / "risk_map.html"); m

## Notes for the slides and next steps

- **Storm channels are not used.** The CSV only has lengths, not locations. To use it, download the same dataset as **GeoJSON** and add distance to the nearest channel with GeoPandas.
- **Sewer overflows only reach crossings whose name includes a creek.** The others get 0 for that feature, which means "unknown", not "none".
- **Radii are judgment calls.** Try `INLET_RADIUS_FT = 1000` and `2500` and check whether the top 10 changes; mention it if it doesn't.
- **Next:** `crossings.csv` goes to the briefs notebook, which sends each row's numbers to Qwen on the GB10.